# Student Feedback Theme Discovery Using Unsupervised NLP

**Project Goal**: Discover, cluster, evaluate, and visualize latent educational themes from unstructured student feedback comments using unsupervised NLP techniques (TF-IDF vs SBERT, K-Means vs HDBSCAN).

---

## 1. Environment Setup & Data Loading

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add root directory to path
sys.path.append('..')

from src.preprocessing import preprocess_dataframe
from src.features import create_tfidf_features, create_embeddings
from src.clustering import run_kmeans, run_hdbscan, extract_cluster_keywords
from src.evaluation import evaluate_clustering
from src.visualization import (
    plot_k_evaluation,
    plot_umap_clusters,
    plot_theme_distribution,
    plot_eda_text_length,
    plot_eda_sentiment_distribution,
    plot_cluster_sentiment_crosstab
)

# Set style
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

# Load raw dataset
df_raw = pd.read_csv('../data/raw/Student_Feedback.csv', encoding='latin1')
print('Raw Dataset Shape:', df_raw.shape)
print('Raw Columns:', df_raw.columns.tolist())
df_raw.head()

## 2. Exploratory Data Analysis (EDA)

We explore the raw dataset distributions (sentiment ratings and raw text properties).

In [ ]:
# Plot Sentiment Distribution EDA
plot_eda_sentiment_distribution(df_raw, save_path='../results/figures/eda_sentiment_distribution.png')
df_raw['sentiments'].value_counts()

## 3. Preprocessing & Text Cleaning

The `sentiments` column is explicitly dropped to guarantee an **unsupervised** theme discovery pipeline.

In [ ]:
df = preprocess_dataframe(df_raw)
print('Cleaned Data Shape:', df.shape)
print('Average Word Count:', df['word_count'].mean())
print('Average Character Count:', df['char_count'].mean())

# Plot Feedback Text Length Distributions
plot_eda_text_length(df, save_path='../results/figures/eda_text_length_distribution.png')
df[['original_text', 'clean_text', 'word_count', 'char_count']].head()

## 4. Feature Representations

- **TF-IDF**: Sparse n-gram matrix (Baseline)
- **SBERT**: Dense 384-dimensional sentence transformer embeddings (`all-MiniLM-L6-v2`)

In [ ]:
texts = df['clean_text'].tolist()

# TF-IDF
X_tfidf, tfidf_vec = create_tfidf_features(texts, max_features=3000, ngram_range=(1, 2))
print('TF-IDF Matrix Shape:', X_tfidf.shape)

# SBERT Embeddings
X_sbert = create_embeddings(texts, model_name='all-MiniLM-L6-v2')
print('SBERT Embedding Matrix Shape:', X_sbert.shape)

## 5. Baseline Model: TF-IDF + K-Means (Hyperparameter Tuning K=2..10)

In [ ]:
k_eval = []
for k in range(2, 11):
    m, l = run_kmeans(X_tfidf, n_clusters=k)
    metrics = evaluate_clustering(X_tfidf, l)
    k_eval.append({'k': k, **metrics})

df_k = pd.DataFrame(k_eval)
plot_k_evaluation(df_k, save_path='../results/figures/silhouette_vs_k.png')
best_k = int(df_k.loc[df_k['silhouette'].idxmax(), 'k'])
print(f'Optimal K selected: {best_k}')
df_k

## 6. Improved Models: SBERT + K-Means & SBERT + HDBSCAN

In [ ]:
# Baseline K-Means
m_tfidf, l_tfidf = run_kmeans(X_tfidf, n_clusters=best_k)
metrics_tfidf = evaluate_clustering(X_tfidf, l_tfidf)

# Improved 1: SBERT K-Means
m_sbert_km, l_sbert_km = run_kmeans(X_sbert, n_clusters=best_k)
metrics_sbert_km = evaluate_clustering(X_sbert, l_sbert_km)

# Improved 2: SBERT HDBSCAN
m_hdb, l_hdb = run_hdbscan(X_sbert, min_cluster_size=15, min_samples=3, use_umap=True)
metrics_hdb = evaluate_clustering(X_sbert, l_hdb)

df['cluster_tfidf_kmeans'] = l_tfidf
df['cluster_sbert_kmeans'] = l_sbert_km
df['cluster_sbert_hdbscan'] = l_hdb

## 7. Comparative Evaluation & Visual Projections

In [ ]:
comp_table = pd.DataFrame([
    {'Model': 'Baseline: TF-IDF + K-Means', **metrics_tfidf},
    {'Model': 'Improved 1: SBERT + K-Means', **metrics_sbert_km},
    {'Model': 'Improved 2: SBERT + HDBSCAN', **metrics_hdb}
])
comp_table

In [ ]:
# Plot UMAP projections
plot_umap_clusters(X_sbert, l_sbert_km, title='SBERT + K-Means UMAP Projection', save_path='../results/figures/umap_clusters_sbert_kmeans.png')
plot_umap_clusters(X_sbert, l_hdb, title='SBERT + HDBSCAN UMAP Projection', save_path='../results/figures/umap_clusters_sbert_hdbscan.png')

## 8. Discovered Themes & Keywords (c-TF-IDF)

In [ ]:
keywords_sbert_km = extract_cluster_keywords(df['clean_text'], l_sbert_km, top_n=8)
for c_id, kws in keywords_sbert_km.items():
    sample = df[df['cluster_sbert_kmeans'] == c_id]['original_text'].iloc[0]
    print(f"Theme {c_id}: {', '.join(kws)}")
    print(f"  Quote: {sample}\n")